In [2]:
import os
import sys
import webbrowser
import base64
app = None
# For production deployment
PORT = int(os.environ.get("PORT", 8050))
HOST = os.environ.get("HOST", "0.0.0.0")

# ===== FIX FOR INTERACTIVE ENVIRONMENTS =====
if not hasattr(sys.modules['__main__'], '__file__'):
    sys.modules['__main__'].__file__ = 'app.py'

os.environ['PLOTLY_OFFLINE_MODE'] = 'True'
# ============================================

import dash
from dash import dcc, html
from dash.dependencies import Input, Output

# Initialize app with suppress_callback_exceptions=True
app = dash.Dash(__name__, title="Risk Dashboard", suppress_callback_exceptions=True)

# Dark color scheme with green accents
DARK_BG = 'black'
CARD_BG = '#161b22'
GREEN_ACCENT = '#00c853'
GREEN_GLOW = '#00e676'
TEXT_LIGHT = '#f0f6fc'
TEXT_MUTED = '#8b949e'

# URLs
RISK_CALC_URL = "http://127.0.0.1:8051"  # <-- CHANGE THIS TO YOUR RISK CALC URL
MAP_FILE = "tanzania_interactive.html"   # Your saved map file

# ==================== APP LAYOUT ====================
app.layout = html.Div([
    # SIDEBAR - Fixed left, full height
    html.Div([
        # Header
        html.Div(" RISK", style={
            'color': GREEN_ACCENT,
            'fontWeight': 'bold',
            'fontSize': '28px',
            'letterSpacing': '1px',
            'textAlign': 'center',
            'paddingBottom': '20px',
            'borderBottom': f'1px solid {GREEN_ACCENT}',
            'marginBottom': '20px'
        }),
        
        # HOME button
        html.Div(" HOME", id='btn-home', className='side-btn', n_clicks=0, style={
            'padding': '14px 20px',
            'margin': '8px 0',
            'color': DARK_BG,
            'backgroundColor': GREEN_ACCENT,
            'borderRadius': '10px',
            'cursor': 'pointer',
            'fontSize': '18px',
            'fontWeight': '600',
            'transition': 'all 0.3s ease',
            'border': f'1px solid {GREEN_GLOW}',
            'textAlign': 'center',
            'boxShadow': f'0 0 20px {GREEN_GLOW}55'
        }),
        
        # MAP button
        html.Div(" MAP", id='btn-map', className='side-btn', n_clicks=0, style={
            'padding': '14px 20px',
            'margin': '8px 0',
            'color': TEXT_LIGHT,
            'backgroundColor': CARD_BG,
            'borderRadius': '10px',
            'cursor': 'pointer',
            'fontSize': '18px',
            'fontWeight': '500',
            'transition': 'all 0.3s ease',
            'border': f'1px solid transparent',
            'textAlign': 'center'
        }),
        
        # RISK CALC button - Opens in new tab
        html.A(
            html.Div(" RISK CALC", className='side-btn', style={
                'padding': '14px 20px',
                'margin': '8px 0',
                'color': TEXT_LIGHT,
                'backgroundColor': CARD_BG,
                'borderRadius': '10px',
                'cursor': 'pointer',
                'fontSize': '18px',
                'fontWeight': '500',
                'transition': 'all 0.3s ease',
                'border': f'1px solid transparent',
                'textAlign': 'center',
                'textDecoration': 'none'
            }),
            href=RISK_CALC_URL,
            target="_blank",
            style={'textDecoration': 'none'}
        ),
        
        # Footer text
        html.P("Click to navigate", style={
            'color': TEXT_MUTED,
            'fontSize': '12px',
            'textAlign': 'center',
            'marginTop': '60px',
            'fontStyle': 'italic'
        })
        
    ], style={
        'width': '220px',
        'height': '100vh',
        'backgroundColor': CARD_BG,
        'padding': '30px 20px',
        'borderRight': f'2px solid {GREEN_ACCENT}',
        'display': 'flex',
        'flexDirection': 'column',
        'position': 'fixed',
        'top': 0,
        'left': 0,
        'bottom': 0,
        'boxShadow': f'4px 0 20px {GREEN_GLOW}22',
        'overflowY': 'auto',
        'zIndex': 1000
    }),
    
    # MAIN CONTENT
    html.Div(id='main-content', style={
        'marginLeft': '220px',
        'padding': '40px 50px',
        'backgroundColor': DARK_BG,
        'minHeight': '100vh',
        'width': 'calc(100% - 220px)',
        'position': 'absolute',
        'top': 0,
        'right': 0,
        'bottom': 0,
        'overflowY': 'auto'
    })
    
], style={
    'margin': '0',
    'padding': '0',
    'width': '100vw',
    'height': '100vh',
    'position': 'fixed',
    'top': 0,
    'left': 0,
    'overflow': 'hidden',
    'backgroundColor': DARK_BG
})

# ========== FUNCTION TO LOAD MAP HTML ==========
def load_map_html():
    """Load the map HTML file and return it as a string"""
    try:
        with open(MAP_FILE, 'r', encoding='utf-8') as f:
            return f.read()
    except FileNotFoundError:
        return f"""
        <div style="padding: 40px; color: #f0f6fc; text-align: center;">
            <h2 style="color: #ff6b6b;">Map File Not Found</h2>
            <p style="color: #8b949e;">Could not find: {MAP_FILE}</p>
            <div style="margin-top: 20px; padding: 20px; background: #161b22; border-radius: 10px;">
                <p> Please run your Jupyter notebook to generate the map first.</p>
                <p style="color: #8b949e; margin-top: 10px;">Make sure <code style="background: #0d1117; padding: 2px 6px; border-radius: 3px;">{MAP_FILE}</code> is in the same folder.</p>
            </div>
        </div>
        """
    except Exception as e:
        return f"""
        <div style="padding: 40px; color: #f0f6fc; text-align: center;">
            <h2 style="color: #ff6b6b;"> Error Loading Map</h2>
            <p style="color: #8b949e;">{str(e)}</p>
        </div>
        """

# ========== CALLBACKS FOR NAVIGATION ==========
@app.callback(
    [Output('btn-home', 'style'),
     Output('btn-map', 'style'),
     Output('main-content', 'children')],
    [Input('btn-home', 'n_clicks'),
     Input('btn-map', 'n_clicks')],
    prevent_initial_call=False
)
def navigate(home_clicks, map_clicks):
    ctx = dash.callback_context
    
    # Default styles
    default_style = {
        'padding': '14px 20px',
        'margin': '8px 0',
        'color': TEXT_LIGHT,
        'backgroundColor': CARD_BG,
        'borderRadius': '10px',
        'cursor': 'pointer',
        'fontSize': '18px',
        'fontWeight': '500',
        'transition': 'all 0.3s ease',
        'border': f'1px solid transparent',
        'textAlign': 'center'
    }
    
    active_style = {
        'padding': '14px 20px',
        'margin': '8px 0',
        'color': DARK_BG,
        'backgroundColor': GREEN_ACCENT,
        'borderRadius': '10px',
        'cursor': 'pointer',
        'fontSize': '18px',
        'fontWeight': '600',
        'transition': 'all 0.3s ease',
        'border': f'1px solid {GREEN_GLOW}',
        'textAlign': 'center',
        'boxShadow': f'0 0 20px {GREEN_GLOW}55'
    }
    
    # If no clicks yet (initial load), show home
    if not ctx.triggered:
        return active_style, default_style, render_home()
    
    # Get which button was clicked
    button_id = ctx.triggered[0]['prop_id'].split('.')[0]
    
    if button_id == 'btn-home':
        return active_style, default_style, render_home()
    elif button_id == 'btn-map':
        return default_style, active_style, render_map()
    
    return default_style, default_style, render_home()

# ========== RENDER FUNCTIONS ==========
def render_home():
    """Render the home page content"""
    return html.Div([
        html.H1(" Welcome to Risk Dashboard", style={
            'color': TEXT_LIGHT,
            'fontWeight': '300',
            'letterSpacing': '1px',
            'marginTop': '0px'
        }),
        html.P("Monitor, visualize, and calculate risk across your locations.", style={
            'color': TEXT_MUTED,
            'fontSize': '18px'
        }),
        
        # Feature cards
        html.Div([
            html.Div(" Real-time monitoring climate risk", style={
                'color': GREEN_ACCENT,
                'padding': '15px',
                'borderBottom': f'1px solid {GREEN_ACCENT}33'
            }),
            html.Div(" Interactive mapping of highly and low risk region", style={
                'color': GREEN_ACCENT,
                'padding': '15px',
                'borderBottom': f'1px solid {GREEN_ACCENT}33'
            }),
            html.Div(" Risk scoring engine for financial institution", style={
                'color': GREEN_ACCENT,
                'padding': '15px'
            })
        ], style={
            'backgroundColor': CARD_BG,
            'borderRadius': '12px',
            'padding': '10px 20px',
            'marginTop': '30px',
            'maxWidth': '500px',
            'border': f'1px solid {GREEN_ACCENT}33'
        }),
        
        # Navigation hints
        html.Div([
            html.P(" Click 'RISK CALC' to open your Risk Calculator", style={
                'color': GREEN_ACCENT,
                'marginTop': '30px',
                'fontSize': '16px'
            }),
            html.P("Click 'MAP' to view your interactive risk map", style={
                'color': TEXT_MUTED,
                'fontSize': '16px'
            })
        ])
    ])

def render_map():
    """Render the map page with the embedded HTML"""
    map_html = load_map_html()
    
    return html.Div([
        # Return to Home button (now with a unique ID that exists when rendered)
        html.Button(
            "← Return to Home", 
            id='return-home-btn',
            style={
                'backgroundColor': CARD_BG,
                'color': GREEN_ACCENT,
                'border': f'1px solid {GREEN_ACCENT}',
                'padding': '10px 20px',
                'borderRadius': '8px',
                'cursor': 'pointer',
                'fontSize': '16px',
                'fontWeight': '500',
                'transition': 'all 0.3s ease',
                'marginBottom': '20px'
            },
            n_clicks=0
        ),
        html.H1(" Tanzania Risk Map", style={
            'color': TEXT_LIGHT,
            'fontWeight': '300',
            'letterSpacing': '1px',
            'marginTop': '0px',
            'marginBottom': '20px'
        }),
        html.Div([
            # Display the map using an iframe with the HTML content
            html.Iframe(
                srcDoc=map_html,
                style={
                    'width': '100%',
                    'height': '700px',
                    'border': f'2px solid {GREEN_ACCENT}33',
                    'borderRadius': '12px',
                    'backgroundColor': CARD_BG
                }
            )
        ], style={
            'backgroundColor': CARD_BG,
            'borderRadius': '12px',
            'padding': '10px',
            'border': f'1px solid {GREEN_ACCENT}33'
        })
    ])

# ========== CALLBACK FOR RETURN TO HOME ==========
@app.callback(
    [Output('btn-home', 'style', allow_duplicate=True),
     Output('btn-map', 'style', allow_duplicate=True),
     Output('main-content', 'children', allow_duplicate=True)],
    Input('return-home-btn', 'n_clicks'),
    prevent_initial_call=True
)
def return_home(n_clicks):
    if n_clicks and n_clicks > 0:
        active_style = {
            'padding': '14px 20px',
            'margin': '8px 0',
            'color': DARK_BG,
            'backgroundColor': GREEN_ACCENT,
            'borderRadius': '10px',
            'cursor': 'pointer',
            'fontSize': '18px',
            'fontWeight': '600',
            'transition': 'all 0.3s ease',
            'border': f'1px solid {GREEN_GLOW}',
            'textAlign': 'center',
            'boxShadow': f'0 0 20px {GREEN_GLOW}55'
        }
        default_style = {
            'padding': '14px 20px',
            'margin': '8px 0',
            'color': TEXT_LIGHT,
            'backgroundColor': CARD_BG,
            'borderRadius': '10px',
            'cursor': 'pointer',
            'fontSize': '18px',
            'fontWeight': '500',
            'transition': 'all 0.3s ease',
            'border': f'1px solid transparent',
            'textAlign': 'center'
        }
        return active_style, default_style, render_home()
    return dash.no_update

# ========== CSS STYLING ==========
app.index_string = '''
<!DOCTYPE html>
<html style="margin:0; padding:0; width:100%; height:100%;">
    <head>
        {%metas%}
        <title>{%title%}</title>
        {%favicon%}
        {%css%}
        <style>
            * {
                margin: 0;
                padding: 0;
                box-sizing: border-box;
            }
            html, body {
                margin: 0 !important;
                padding: 0 !important;
                width: 100% !important;
                height: 100% !important;
                overflow: hidden !important;
                font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            }
            #react-entry {
                width: 100%;
                height: 100%;
            }
            
            /* Sidebar button hover effects */
            .side-btn:hover {
                background-color: #00c853 !important;
                color: #0d1117 !important;
                border-color: #00e676 !important;
                box-shadow: 0 0 20px #00e67655 !important;
                transform: translateX(6px) scale(1.02);
                font-weight: 600 !important;
            }
            .side-btn:active {
                transform: scale(0.97);
            }
            
            /* Return to home button hover */
            #return-home-btn:hover {
                background-color: #00c853 !important;
                color: #0d1117 !important;
                box-shadow: 0 0 20px #00e67655 !important;
                transform: scale(1.02);
            }
            
            /* Scrollbar styling */
            ::-webkit-scrollbar {
                width: 8px;
            }
            ::-webkit-scrollbar-track {
                background: #0d1117;
            }
            ::-webkit-scrollbar-thumb {
                background: #00c853;
                border-radius: 4px;
            }
            ::-webkit-scrollbar-thumb:hover {
                background: #00e676;
            }
            
            /* Folium map styling within iframe */
            .leaflet-container {
                background: #0d1117 !important;
            }
            .leaflet-popup-content-wrapper {
                background: #161b22 !important;
                color: #f0f6fc !important;
                border-radius: 8px !important;
                border: 1px solid #00c853 !important;
            }
            .leaflet-popup-tip {
                background: #161b22 !important;
            }
            
            /* Link styling */
            a {
                text-decoration: none;
            }
        </style>
    </head>
    <body style="margin:0; padding:0; width:100%; height:100%; overflow:hidden;">
        {%app_entry%}
        <footer>
            {%config%}
            {%scripts%}
            {%renderer%}
        </footer>
    </body>
</html>
'''

# ========== RUN THE APP ==========
if __name__ == '__main__':
    print("\n" + "="*50)
    print(" RISK DASHBOARD STARTING...")
    print("="*50)
    print(f" Dashboard URL: http://127.0.0.1:8050")
    print(f" Map file: {MAP_FILE}")
    print(f" Risk Calc URL: {RISK_CALC_URL}")
    print("="*50)
    print("\n Make sure 'tanzania_interactive.html' exists in the same folder")
    print("   If not, run your Jupyter notebook to generate it first.")
    print("\nPress Ctrl+C to stop the server\n")
    PORT = int(os.environ.get("PORT", 8050))
    app.run(host='0.0.0.0',debug=False, port=PORT, use_reloader=False)


 RISK DASHBOARD STARTING...
 Dashboard URL: http://127.0.0.1:8050
 Map file: tanzania_interactive.html
 Risk Calc URL: http://127.0.0.1:8051

 Make sure 'tanzania_interactive.html' exists in the same folder
   If not, run your Jupyter notebook to generate it first.

Press Ctrl+C to stop the server



ConnectionError: HTTPConnectionPool(host='0.0.0.0', port=8050): Max retries exceeded with url: /_alive_897b534b-6244-41d7-b861-818474492ba7 (Caused by NewConnectionError("HTTPConnection(host='0.0.0.0', port=8050): Failed to establish a new connection: [WinError 10049] The requested address is not valid in its context"))